In [6]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [7]:
EXCHANGE_RATE_KEY = os.getenv('EXCHANGE_RATE_API_KEY')

In [8]:
from langchain_core.tools import tool
import requests

@tool(
    description="""
    Get the currency conversion rate between a base currency and target currency.
    Use this tool first when a user asks to convert currencies. Pass its
    conversion rate result to the convert tool.
    """
)
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    
    url = f'https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_KEY}/pair/{base_currency}/{target_currency}'
    
    raw_result = requests.get(url)
    data = raw_result.json()
    return data['conversion_rate']

@tool(
    description="""
    Convert a currency amount using a conversion rate obtained from
    get_conversion_factor. Use this tool after getting the conversion rate.
    """
)
def convert(base_currency_value: float, conversion_rate: float) -> float:
    return base_currency_value * conversion_rate

In [36]:
conversion_rate = get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'INR'})
convert.invoke({'base_currency_value': 80,'conversion_rate':conversion_rate})

7561.408

In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(model='openai/gpt-oss-120b',max_tokens=200,temperature=0.1)

llm_with_tools = llm.bind_tools([get_conversion_factor,convert])
print(llm_with_tools.kwargs)

{'tools': [{'type': 'function', 'function': {'name': 'get_conversion_factor', 'description': 'Get the currency conversion rate between a base currency and target currency.\n    Use this tool first when a user asks to convert currencies. Pass its\n    conversion rate result to the convert tool.', 'parameters': {'properties': {'base_currency': {'type': 'string'}, 'target_currency': {'type': 'string'}}, 'required': ['base_currency', 'target_currency'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'convert', 'description': 'Convert a currency amount using a conversion rate obtained from\n    get_conversion_factor. Use this tool after getting the conversion rate.', 'parameters': {'properties': {'base_currency_value': {'type': 'number'}, 'conversion_rate': {'type': 'number'}}, 'required': ['base_currency_value', 'conversion_rate'], 'type': 'object'}}}]}


In [11]:
def execute_tool(tool_call):
    print(f'Tool Call {tool_call}')
    if tool_call['name'] == 'get_conversion_factor':
        return get_conversion_factor.invoke(tool_call)
    if tool_call['name'] == 'convert':
        return convert.invoke(tool_call)

In [ ]:
from langchain_core.messages import HumanMessage
query = HumanMessage('What is 20 dollars equivalent to indian rupee?')
messages = [query];

response = llm_with_tools.invoke(messages)

while response.tool_calls:
    for tool_call in response.tool_calls:
        tool_result = execute_tool(tool_call)
        
        messages.append(response)
        messages.append(tool_result)
    
    response = llm_with_tools.invoke(messages)

print(response.content)

{'reasoning_content': 'User asks: "What is 20 dollars equivalent to indian rupee?" Need to get conversion rate USD to INR. Use get_conversion_factor then convert.', 'tool_calls': [{'id': 'fc_711f02e1-ff4b-40ef-91c9-6eff0606b4c2', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}]}
Tool Call {'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'fc_711f02e1-ff4b-40ef-91c9-6eff0606b4c2', 'type': 'tool_call'}
{'reasoning_content': 'We have conversion rate: 1 USD = 94.5176 INR. Need to convert 20 USD to INR: 20 * 94.5176 = 1890.352. Use convert tool.', 'tool_calls': [{'id': 'fc_8bd14fb7-fdf1-4e99-a98e-8b5922dc5f14', 'function': {'arguments': '{"base_currency_value":20,"conversion_rate":94.5176}', 'name': 'convert'}, 'type': 'function'}]}
Tool Call {'name': 'convert', 'args': {'base_currency_value': 20, 'conversion_rate': 94.5176}, 'id': 'fc_8bd14fb7-fdf1-4e99-a98e-